# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their `@id`.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL (metadata)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# mlcroissant v0.0.14+ (2024) metadata as a rich object:
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields, referencing all entities by their `@id`.

In [ ]:
# List all record sets by their @id and name

print("Available Record Sets in the Dataset:\n==============================")
for record_set in metadata.record_sets:
    print(f"@id: {record_set['@id']}, name: {record_set.get('name', '<no name>')}")
    # List fields for each record set
    if 'field' in record_set:
        fields = record_set['field']
        if not isinstance(fields, list):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            if isinstance(field, dict):
                fid = field.get('@id', '<no id>')
                fname = field.get('name', '<no name>')
            else:
                fid = field
                fname = ''
            print(f"    @id: {fid}\t{name}")
    print("---")

### Select a Record Set
Choose a record set. For this dataset the main record set typically contains the actual tabular data, e.g., protein analysis table. Obtain its `@id` from above. In this example, we will use `'cr:RecordSet1'` and its numeric field `cr:peptide_count_total` for demonstrations. **Replace with the actual `@id` from the above output if needed**.

In [ ]:
# Example: list record_set @ids (replace below as needed)
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

# For demonstration, select the main data table record set
# (If you have multiple record sets, choose by name, here we simply use the first)
main_record_set_id = record_set_ids[0]  # Update this to correct '@id' if different
print(f"Selected record set: {main_record_set_id}")

## 3. Data Extraction
Load data from the specific record set into a DataFrame, referencing the record set and field `@id`s from the overview cell above.

In [ ]:
dataframes = {}
# For each record set, load the data
for record_set_id in record_set_ids:
    print(f"Loading records from {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only store non-empty
        dataframes[record_set_id] = pd.DataFrame(records)

if main_record_set_id in dataframes:
    print(f"Fields (columns) in {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"No records found for record set {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

All fields are referenced by their `@id`s.

In [ ]:
# Pick a numeric field and (optionally) a categorical field by their @id
# Example field ids (replace as appropriate from your overview output)
numeric_field_id = None  # E.g., 'cr:peptide_count_total'
group_field_id = None    # E.g., 'cr:modification_type'

# Try to auto-pick numeric/categorical fields
if main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    for col in df.columns:
        if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break

if not numeric_field_id:
    raise ValueError("Could not auto-detect a numeric field. Please set numeric_field_id explicitly from available columns.")
if not group_field_id:
    print("Could not auto-detect a group field. Grouping will be skipped.")

threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()

print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All columns use their original `@id` names.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id in dataframes and numeric_field_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of `{numeric_field_id}`")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset using `mlcroissant`, referencing all record sets and fields by their `@id`. We filtered, normalized, and visualized data using common data science tools. 

Key findings and further directions will depend on specific domain analysis. For protein datasets, users might investigate patterns in abundance and modification fields or compare between experimental conditions. Please refer to the dataset's Croissant schema for further field details and relationships.